**Downoald GNSS data drom GEIN NOA archive service**

In [ ]:
# CELL 2 — User-selected download path and calendar date range for DELO RINEX files

from pathlib import Path
from datetime import datetime, timedelta
import requests

BASE_URL = "https://www.gein.noa.gr/services/GPSData"
STATION = "DELO00GRC"

# --- User input ---
download_path_str = input(
    "Enter download folder path "
    "(e.g. /content/drive/MyDrive/RINEX/DELO or rinex_DELO): "
).strip()

start_date_str = input("Enter start date [YYYY-MM-DD]: ").strip()
end_date_str   = input("Enter end date   [YYYY-MM-DD]: ").strip()

# --- Parse and validate input ---
if not download_path_str:
    raise ValueError("Download folder path cannot be empty.")

try:
    START_DATE = datetime.strptime(start_date_str, "%Y-%m-%d").date()
    END_DATE   = datetime.strptime(end_date_str, "%Y-%m-%d").date()
except ValueError:
    raise ValueError("Dates must be given in YYYY-MM-DD format.")

if END_DATE < START_DATE:
    raise ValueError("End date must be equal to or later than start date.")

OUT_DIR = Path(download_path_str).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nDownload configuration")
print(f"Station:      {STATION}")
print(f"Start date:   {START_DATE}")
print(f"End date:     {END_DATE}")
print(f"Output path:  {OUT_DIR.resolve()}")

# --- Download loop ---
downloaded = []
already_existing = []
missing = []
failed = []

current_date = START_DATE

while current_date <= END_DATE:
    year = current_date.year
    doy = current_date.timetuple().tm_yday

    filename = f"{STATION}_R_{year}{doy:03d}0000_01D_30S_MO.crx.gz"
    url = f"{BASE_URL}/{year}/{doy:03d}/{filename}"
    out_file = OUT_DIR / filename

    if out_file.exists() and out_file.stat().st_size > 0:
        already_existing.append(filename)
        print(f"Already exists: {filename}")
        current_date += timedelta(days=1)
        continue

    try:
        response = requests.get(url, timeout=60)

        if response.status_code == 200:
            out_file.write_bytes(response.content)

            if out_file.stat().st_size > 0:
                downloaded.append(filename)
                print(f"Downloaded: {filename}")
            else:
                failed.append((filename, "empty file"))
                print(f"Failed: {filename} | empty file")

        elif response.status_code == 404:
            missing.append(filename)
            print(f"Missing 404: {filename}")

        else:
            failed.append((filename, response.status_code))
            print(f"Failed: {filename} | HTTP {response.status_code}")

    except Exception as e:
        failed.append((filename, str(e)))
        print(f"Error: {filename} | {e}")

    current_date += timedelta(days=1)

# --- Summary ---
print("\nSummary")
print(f"Downloaded:        {len(downloaded)}")
print(f"Already existing:  {len(already_existing)}")
print(f"Missing 404:       {len(missing)}")
print(f"Failed:            {len(failed)}")
print(f"Output folder:     {OUT_DIR.resolve()}")

In [ ]:
# CELL 3 — Automatic unzip of downloaded *.gz RINEX files

from pathlib import Path
import gzip
import shutil

# Use OUT_DIR from CELL 2 if it exists.
# Otherwise ask the user to provide the folder containing the *.gz files.
if "OUT_DIR" in globals():
    gz_folder = Path(OUT_DIR)
else:
    gz_folder = Path(
        input("Enter folder path containing the *.gz files: ").strip()
    ).expanduser()

if not gz_folder.exists():
    raise FileNotFoundError(f"Folder does not exist: {gz_folder}")

gz_files = sorted(gz_folder.glob("*.gz"))

if not gz_files:
    print(f"No *.gz files found in: {gz_folder.resolve()}")
else:
    unzipped = []
    already_existing = []
    failed = []

    for gz_file in gz_files:
        out_file = gz_file.with_suffix("")  # removes only the final .gz

        if out_file.exists() and out_file.stat().st_size > 0:
            already_existing.append(out_file.name)
            print(f"Already exists: {out_file.name}")
            continue

        try:
            with gzip.open(gz_file, "rb") as f_in:
                with open(out_file, "wb") as f_out:
                    shutil.copyfileobj(f_in, f_out)

            if out_file.stat().st_size > 0:
                unzipped.append(out_file.name)
                print(f"Unzipped: {gz_file.name} -> {out_file.name}")
            else:
                failed.append((gz_file.name, "empty output file"))
                print(f"Failed: {gz_file.name} | empty output file")

        except Exception as e:
            failed.append((gz_file.name, str(e)))
            print(f"Error: {gz_file.name} | {e}")

    print("\nSummary")
    print(f"GZ files found:    {len(gz_files)}")
    print(f"Unzipped:          {len(unzipped)}")
    print(f"Already existing:  {len(already_existing)}")
    print(f"Failed:            {len(failed)}")
    print(f"Folder:            {gz_folder.resolve()}")

In [ ]:
# CELL 4 — Organize unzipped RINEX files into daily dataset folders
#
# Input:
#   RAW_ROOT contains unzipped *.crx files directly inside it
#
# Output:
#   RAW_ROOT/YYYY_DDD/<file>
#
# Example:
#   RAW_ROOT/DELO00GRC_R_20252700000_01D_30S_MO.crx
#   ->
#   RAW_ROOT/2025_270/DELO00GRC_R_20252700000_01D_30S_MO.crx

from pathlib import Path
import re
import shutil

# Use OUT_DIR from previous cells if available.
# Otherwise ask for RAW_ROOT manually.
if "OUT_DIR" in globals():
    RAW_ROOT = Path(OUT_DIR)
else:
    RAW_ROOT = Path(
        input("Enter RAW_ROOT folder path: ").strip()
    ).expanduser()

if not RAW_ROOT.exists():
    raise FileNotFoundError(f"RAW_ROOT does not exist: {RAW_ROOT}")

# RINEX filename pattern:
# DELO00GRC_R_20252700000_01D_30S_MO.crx
rinex_pattern = re.compile(
    r"^(?P<station>[A-Z0-9]{9})_R_(?P<year>\d{4})(?P<doy>\d{3})0000_01D_30S_MO\.crx$"
)

def make_dataset_name(year: int, doy: int) -> str:
    """
    Daily dataset folder naming convention.
    Current convention: YYYY_DDD
    Example: 2025_270
    """
    return f"{year}_{doy:03d}"

crx_files = sorted(RAW_ROOT.glob("*.crx"))

if not crx_files:
    print(f"No *.crx files found directly inside: {RAW_ROOT.resolve()}")
else:
    moved = []
    already_in_place = []
    skipped = []
    failed = []

    for crx_file in crx_files:
        match = rinex_pattern.match(crx_file.name)

        if not match:
            skipped.append(crx_file.name)
            print(f"Skipped, filename does not match expected pattern: {crx_file.name}")
            continue

        year = int(match.group("year"))
        doy = int(match.group("doy"))

        dataset_name = make_dataset_name(year, doy)
        dataset_dir = RAW_ROOT / dataset_name
        dataset_dir.mkdir(parents=True, exist_ok=True)

        target_file = dataset_dir / crx_file.name

        if target_file.exists() and target_file.stat().st_size > 0:
            already_in_place.append(crx_file.name)
            print(f"Already exists in dataset folder: {dataset_name}/{crx_file.name}")
            continue

        try:
            shutil.move(str(crx_file), str(target_file))
            moved.append(str(target_file.relative_to(RAW_ROOT)))
            print(f"Moved: {crx_file.name} -> {dataset_name}/{crx_file.name}")

        except Exception as e:
            failed.append((crx_file.name, str(e)))
            print(f"Error moving {crx_file.name}: {e}")

    print("\nSummary")
    print(f"CRX files found directly in RAW_ROOT: {len(crx_files)}")
    print(f"Moved:                            {len(moved)}")
    print(f"Already existing:                 {len(already_in_place)}")
    print(f"Skipped:                          {len(skipped)}")
    print(f"Failed:                           {len(failed)}")
    print(f"RAW_ROOT:                         {RAW_ROOT.resolve()}")

In [ ]:
# CELL 5 — Build station coverage inventory from NOA GEIN GPSData repository
#
# Output:
#   station_coverage.csv
#
# Columns:
#   station
#   first_date
#   last_date
#   n_available_days
#   n_files
#   first_dataset_name
#   last_dataset_name
#   example_file

from pathlib import Path
from datetime import date, timedelta
from urllib.parse import urljoin
import re
import time
import requests
import pandas as pd

BASE_URL = "https://www.gein.noa.gr/services/GPSData/"

# --- User input ---
out_path_str = input(
    "Enter output folder path for station_coverage.csv "
    "(e.g. /content/drive/MyDrive/RINEX_INDEX or rinex_index): "
).strip()

start_year_str = input("Enter start year [e.g. 2006]: ").strip()
end_year_str   = input("Enter end year   [e.g. 2026]: ").strip()

if not out_path_str:
    raise ValueError("Output folder path cannot be empty.")

START_YEAR = int(start_year_str)
END_YEAR   = int(end_year_str)

if END_YEAR < START_YEAR:
    raise ValueError("End year must be equal to or later than start year.")

OUT_DIR = Path(out_path_str).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = OUT_DIR / "station_coverage.csv"

# --- Helpers ---
href_pattern = re.compile(r'href="([^"]+)"', re.IGNORECASE)

rinex3_pattern = re.compile(
    r"^(?P<station>[A-Z0-9]{9})_R_\d{7}0000_01D_30S_[MG]O\.crx\.gz$",
    re.IGNORECASE
)

rinex2_pattern = re.compile(
    r"^(?P<station>[A-Z0-9]{4})\d{3}0\.\d{2}[do]\.Z$",
    re.IGNORECASE
)

def fetch_index(url, timeout=60):
    response = requests.get(url, timeout=timeout)
    if response.status_code != 200:
        return None
    return response.text

def extract_hrefs(html):
    return href_pattern.findall(html)

def doy_to_date(year, doy):
    return date(year, 1, 1) + timedelta(days=doy - 1)

def make_dataset_name(year, doy):
    return f"{year}_{doy:03d}"

records = []

print("\nScanning NOA GEIN GPSData repository")
print(f"Years: {START_YEAR}–{END_YEAR}")
print(f"Output CSV: {OUT_CSV.resolve()}\n")

for year in range(START_YEAR, END_YEAR + 1):
    year_url = urljoin(BASE_URL, f"{year}/")
    year_html = fetch_index(year_url)

    if year_html is None:
        print(f"Year not accessible: {year}")
        continue

    doy_dirs = []
    for href in extract_hrefs(year_html):
        if re.fullmatch(r"\d{3}/", href):
            doy_dirs.append(int(href[:3]))

    doy_dirs = sorted(set(doy_dirs))

    print(f"{year}: {len(doy_dirs)} day folders")

    for doy in doy_dirs:
        day_url = urljoin(year_url, f"{doy:03d}/")
        day_html = fetch_index(day_url)

        if day_html is None:
            continue

        current_date = doy_to_date(year, doy)
        dataset_name = make_dataset_name(year, doy)

        for href in extract_hrefs(day_html):
            filename = href.split("/")[-1]

            m3 = rinex3_pattern.match(filename)
            m2 = rinex2_pattern.match(filename)

            if m3:
                station = m3.group("station").upper()
            elif m2:
                station = m2.group("station").upper()
            else:
                continue

            records.append({
                "station": station,
                "date": current_date,
                "year": year,
                "doy": doy,
                "dataset_name": dataset_name,
                "filename": filename,
                "url": urljoin(day_url, filename),
            })

        time.sleep(0.05)

if not records:
    raise RuntimeError("No RINEX files were found in the selected year range.")

df_files = pd.DataFrame(records)

df_coverage = (
    df_files
    .groupby("station", as_index=False)
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        n_available_days=("date", "nunique"),
        n_files=("filename", "count"),
        first_dataset_name=("dataset_name", "min"),
        last_dataset_name=("dataset_name", "max"),
        example_file=("filename", "first"),
    )
    .sort_values(["station"])
    .reset_index(drop=True)
)

df_coverage.to_csv(OUT_CSV, index=False)

print("\nSummary")
print(f"Stations found: {len(df_coverage)}")
print(f"Files indexed:  {len(df_files)}")
print(f"Saved CSV:      {OUT_CSV.resolve()}")

display(df_coverage)